# Clase 2 — Ejercicios de recolección de datos
**Taller de Ciencia de Datos (EIN146A25) · 2026-2 · Jueves 13 de agosto**

Tres ejercicios, uno por vía de acceso: **archivo plano (CSV)**, **base de datos (SQL)** y **API pública**. Al final, la plantilla de *obtención reproducible* que deben aplicar a su propio dataset hoy mismo (bloques 5–6).

Reglas del notebook:
- Los archivos de práctica (CSV y base sqlite) **se generan solos** en la carpeta `data_practica/` — no necesitan descargar nada para los Ejercicios 1 y 2.
- Las celdas marcadas **✏️ TU TURNO** son las que deben completar.
- Cada ejercicio trae una **solución propuesta** más abajo: intenten primero sin mirarla.


In [ ]:
# Setup — ejecutar primero
import sys
from pathlib import Path

import pandas as pd

CARPETA = Path("data_practica")
CARPETA.mkdir(exist_ok=True)

print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)
print("Carpeta de trabajo:", CARPETA.resolve())

---
## Ejercicio 1 — CSV con trampas (≈25 min)

La celda siguiente genera `data_practica/ventas_farmacia.csv`, un CSV **idéntico en espíritu a los que publican los portales chilenos**: codificación `latin-1`, separador `;`, decimales con coma, filas de notas antes del encabezado, nulos escritos como `S/I` y códigos de comuna con cero a la izquierda.

Su misión: leerlo **correctamente** con `pd.read_csv`.

In [ ]:
# Genera el CSV "sucio" (no modificar esta celda)
contenido = """REPORTE DE VENTAS - CADENA DE FARMACIAS
Fuente: Sistema interno de fidelización; Corte: 30-06-2026
;;;;;
id_cliente;codigo_comuna;comuna;n_compras_6m;monto_total;dias_desde_ultima_compra
C0001;05101;Valparaíso;12;145.320,50;8
C0002;13101;Santiago;3;38.900,00;95
C0003;05109;Viña del Mar;S/I;12.450,75;41
C0004;13123;Providencia;25;312.780,00;2
C0005;05101;Valparaíso;7;S/I;33
C0006;08101;Concepción;1;5.990,00;180
C0007;13114;Las Condes;18;S/I;5
C0008;05804;Quilpué;9;87.640,25;-
C0009;13101;Santiago;S/I;S/I;150
C0010;05601;San Antonio;4;41.200,00;61
"""
ruta_csv = CARPETA / "ventas_farmacia.csv"
ruta_csv.write_bytes(contenido.encode("latin-1"))
print("Archivo creado:", ruta_csv, f"({ruta_csv.stat().st_size} bytes)")

### 1.1 Primero, el error
Ejecuten la lectura ingenua y observen **qué** falla y **cómo** falla. Este es el comportamiento normal frente a un archivo real.

In [ ]:
# Lectura ingenua — se espera que falle o entregue basura
try:
    df_malo = pd.read_csv(ruta_csv)
    display(df_malo.head())
    print("No lanzó error... pero miren la forma del resultado:", df_malo.shape)
except Exception as e:
    print(f"💥 {type(e).__name__}: {e}")

### 1.2 Miren el archivo crudo
**Regla de la Clase 2:** antes de `read_csv`, mirar las primeras líneas tal como están en el disco.

In [ ]:
with open(ruta_csv, "rb") as f:
    for i, linea in enumerate(f):
        print(i, repr(linea))
        if i >= 6:
            break

### 1.3 ✏️ TU TURNO
Completen la llamada a `read_csv` para que el resultado cumpla **todo** esto:

1. 10 filas × 6 columnas, con los nombres de columna correctos.
2. `codigo_comuna` como **texto** (el `05101` de Valparaíso debe conservar su cero).
3. `n_compras_6m` y `dias_desde_ultima_compra` numéricos, con los `S/I` y `-` convertidos a `NaN`.
4. `monto_total` numérico (float): `145.320,50` debe quedar como `145320.5`.

Pistas: `encoding`, `sep`, `skiprows`, `decimal`, `thousands`, `dtype`, `na_values`.

In [ ]:
# ✏️ TU TURNO — completen los parámetros
# df = pd.read_csv(
#     ruta_csv,
#     ...
# )
# df.info()
# df.head(10)

### ✅ Verificación automática
Ejecuten esta celda cuando tengan su `df`. Debe imprimir 4 ✓.

In [ ]:
def verificar_ej1(df):
    checks = [
        ("Forma 10×6 y columnas correctas",
         df.shape == (10, 6) and list(df.columns) == [
             "id_cliente", "codigo_comuna", "comuna",
             "n_compras_6m", "monto_total", "dias_desde_ultima_compra"]),
        ("codigo_comuna es texto y conserva el cero (05101)",
         pd.api.types.is_string_dtype(df["codigo_comuna"]) and (df["codigo_comuna"] == "05101").sum() == 2),
        ("S/I y '-' quedaron como NaN (6 nulos en total)",
         int(df[["n_compras_6m", "monto_total", "dias_desde_ultima_compra"]].isna().sum().sum()) == 6),
        ("monto_total es float y el primero vale 145320.5",
         df["monto_total"].dtype == float and abs(df["monto_total"].iloc[0] - 145320.5) < 1e-6),
    ]
    for nombre, ok in checks:
        print(("✓" if ok else "✗"), nombre)
    return all(ok for _, ok in checks)

# verificar_ej1(df)

<details>
<summary><b>Solución propuesta (Ejercicio 1)</b> — clic para desplegar</summary>

```python
df = pd.read_csv(
    ruta_csv,
    encoding="latin-1",
    sep=";",
    skiprows=3,
    decimal=",",
    thousands=".",
    dtype={"codigo_comuna": str},
    na_values=["S/I", "-"],
)
verificar_ej1(df)
```
</details>

In [ ]:
# (Celda del profesor / autocomprobación del notebook)
df_sol1 = pd.read_csv(
    ruta_csv, encoding="latin-1", sep=";", skiprows=3,
    decimal=",", thousands=".", dtype={"codigo_comuna": str},
    na_values=["S/I", "-"],
)
assert verificar_ej1(df_sol1), "La solución de referencia del Ej. 1 no pasa la verificación"
df_sol1.info()

---
## Ejercicio 2 — SQL con sqlite (≈20 min)

Ahora los mismos datos viven en una **base relacional** con dos tablas: `clientes` y `ventas` (transacciones individuales). La celda siguiente crea `data_practica/farmacia.db`.

Criterio de la clase: **reducción de volumen en SQL, análisis fino en pandas.**

In [ ]:
# Crea la base sqlite (no modificar esta celda)
import sqlite3

ruta_db = CARPETA / "farmacia.db"
ruta_db.unlink(missing_ok=True)

con = sqlite3.connect(ruta_db)
cur = con.cursor()
cur.executescript('''
CREATE TABLE clientes (
    id_cliente TEXT PRIMARY KEY,
    comuna TEXT,
    fecha_registro TEXT
);
CREATE TABLE ventas (
    id_venta INTEGER PRIMARY KEY,
    id_cliente TEXT REFERENCES clientes(id_cliente),
    fecha TEXT,
    monto INTEGER,
    n_productos INTEGER
);
''')
clientes = [
    ("C0001", "Valparaíso", "2023-04-11"), ("C0002", "Santiago", "2024-01-20"),
    ("C0003", "Viña del Mar", "2022-11-02"), ("C0004", "Providencia", "2021-06-30"),
    ("C0005", "Valparaíso", "2024-09-15"), ("C0006", "Concepción", "2023-02-08"),
    ("C0007", "Las Condes", "2020-12-01"), ("C0008", "Quilpué", "2024-03-19"),
    ("C0009", "Santiago", "2022-07-27"), ("C0010", "San Antonio", "2023-10-05"),
]
ventas = [
    (1, "C0001", "2026-01-15", 12990, 2), (2, "C0001", "2026-02-03", 8450, 1),
    (3, "C0001", "2026-03-22", 23990, 4), (4, "C0002", "2026-01-08", 5990, 1),
    (5, "C0003", "2026-02-14", 15490, 2), (6, "C0004", "2026-01-30", 45990, 6),
    (7, "C0004", "2026-02-28", 31200, 3), (8, "C0004", "2026-03-30", 27890, 3),
    (9, "C0005", "2026-03-05", 9990, 1), (10, "C0006", "2026-01-02", 5990, 1),
    (11, "C0007", "2026-02-09", 78400, 9), (12, "C0007", "2026-03-11", 54300, 5),
    (13, "C0008", "2026-01-25", 18700, 2), (14, "C0008", "2026-03-08", 22100, 3),
    (15, "C0010", "2026-02-20", 41200, 5), (16, "C0004", "2025-12-15", 19990, 2),
    (17, "C0007", "2025-11-30", 36500, 4), (18, "C0009", "2025-10-12", 7990, 1),
]
cur.executemany("INSERT INTO clientes VALUES (?,?,?)", clientes)
cur.executemany("INSERT INTO ventas VALUES (?,?,?,?,?)", ventas)
con.commit()
print("Base creada:", ruta_db)
print("Tablas:", [r[0] for r in cur.execute("SELECT name FROM sqlite_master WHERE type='table'")])

### 2.1 Calentamiento — leer una tabla completa

In [ ]:
pd.read_sql("SELECT * FROM ventas LIMIT 5", con)

### 2.2 ✏️ TU TURNO
Escriban **una sola consulta SQL** (vía `pd.read_sql`) que entregue, **solo para ventas de 2026**, una fila por comuna con: `comuna`, `n_ventas`, `monto_total`, `monto_promedio`, ordenado por `monto_total` descendente.

Pistas: `JOIN`, `WHERE fecha >= '2026-01-01'`, `GROUP BY`, `ORDER BY`, funciones `COUNT/SUM/AVG`.

**Pregunta para discutir en la puesta en común:** ¿qué parte de esto habría sido mejor hacer en pandas si la tabla tuviera 50 millones de filas? ¿Y si tuviera 500 filas?

In [ ]:
# ✏️ TU TURNO
# resumen = pd.read_sql('''
#     ...
# ''', con)
# resumen

<details>
<summary><b>Solución propuesta (Ejercicio 2)</b> — clic para desplegar</summary>

```python
resumen = pd.read_sql('''
    SELECT c.comuna,
           COUNT(*)      AS n_ventas,
           SUM(v.monto)  AS monto_total,
           AVG(v.monto)  AS monto_promedio
    FROM ventas v
    JOIN clientes c ON c.id_cliente = v.id_cliente
    WHERE v.fecha >= '2026-01-01'
    GROUP BY c.comuna
    ORDER BY monto_total DESC
''', con)
resumen
```
</details>

In [ ]:
# (Celda del profesor / autocomprobación del notebook)
resumen_sol = pd.read_sql('''
    SELECT c.comuna,
           COUNT(*)      AS n_ventas,
           SUM(v.monto)  AS monto_total,
           AVG(v.monto)  AS monto_promedio
    FROM ventas v
    JOIN clientes c ON c.id_cliente = v.id_cliente
    WHERE v.fecha >= '2026-01-01'
    GROUP BY c.comuna
    ORDER BY monto_total DESC
''', con)
assert len(resumen_sol) == 8 and resumen_sol.iloc[0]["comuna"] == "Las Condes"
assert int(resumen_sol["n_ventas"].sum()) == 15  # las 3 ventas de 2025 quedan fuera
resumen_sol

In [ ]:
con.close()  # la conexión es un recurso: se cierra

---
## Ejercicio 3 — API pública: Mindicador.cl (≈15 min)

[Mindicador.cl](https://mindicador.cl) entrega indicadores económicos chilenos en JSON, **sin registro ni API key** — ideal para practicar. Documentación: `https://mindicador.cl/api` · serie anual: `https://mindicador.cl/api/{indicador}/{año}`.

> Si el WiFi del laboratorio falla, la celda usa automáticamente una **muestra de respaldo embebida**: la mecánica JSON → DataFrame es idéntica.

In [ ]:
import json
import requests

URL = "https://mindicador.cl/api/uf/2025"

RESPALDO = {
    "codigo": "uf", "nombre": "Unidad de fomento (UF)", "unidad_medida": "Pesos",
    "serie": [
        {"fecha": "2025-12-31T03:00:00.000Z", "valor": 39643.61},
        {"fecha": "2025-09-30T03:00:00.000Z", "valor": 39383.07},
        {"fecha": "2025-06-30T03:00:00.000Z", "valor": 39184.96},
        {"fecha": "2025-03-31T03:00:00.000Z", "valor": 38713.02},
        {"fecha": "2025-01-02T03:00:00.000Z", "valor": 38416.69},
    ],
}

try:
    r = requests.get(URL, timeout=15)
    r.raise_for_status()          # lanza error si la API respondió 4xx/5xx
    datos = r.json()
    origen = "API en vivo"
except Exception as e:
    print(f"⚠️  Sin conexión con la API ({type(e).__name__}). Usando muestra de respaldo.")
    datos = RESPALDO
    origen = "respaldo embebido"

print("Origen:", origen)
print("Indicador:", datos["nombre"], "| puntos en la serie:", len(datos["serie"]))

### 3.1 Regla de oro con APIs: guardar la respuesta cruda
La API de hoy puede no responder lo mismo mañana (o no responder). El crudo va a disco **antes** de cualquier transformación.

In [ ]:
ruta_json = CARPETA / "uf_2025_crudo.json"
ruta_json.write_text(json.dumps(datos, ensure_ascii=False, indent=2), encoding="utf-8")
print("Respuesta cruda guardada en:", ruta_json)

### 3.2 ✏️ TU TURNO
1. Conviertan `datos["serie"]` en un DataFrame `uf`.
2. Transformen `fecha` a tipo datetime (`pd.to_datetime`) y ordenen de más antigua a más reciente.
3. Respondan con código: ¿cuánto **varió porcentualmente** la UF entre el primer y el último valor de la serie?

**Bonus** (si van adelantados): repitan con otro indicador (`dolar`, `ipc`, `bitcoin`) o grafiquen la serie con `uf.plot(x="fecha", y="valor")`.

In [ ]:
# ✏️ TU TURNO
# uf = ...
# variacion = ...
# print(f"Variación: {variacion:.2f}%")

<details>
<summary><b>Solución propuesta (Ejercicio 3)</b> — clic para desplegar</summary>

```python
uf = pd.DataFrame(datos["serie"])
uf["fecha"] = pd.to_datetime(uf["fecha"])
uf = uf.sort_values("fecha").reset_index(drop=True)

variacion = (uf["valor"].iloc[-1] / uf["valor"].iloc[0] - 1) * 100
print(f"Variación: {variacion:.2f}%")
```
</details>

In [ ]:
# (Celda del profesor / autocomprobación del notebook)
uf_sol = pd.DataFrame(datos["serie"])
uf_sol["fecha"] = pd.to_datetime(uf_sol["fecha"])
uf_sol = uf_sol.sort_values("fecha").reset_index(drop=True)
assert uf_sol["fecha"].is_monotonic_increasing and {"fecha", "valor"} <= set(uf_sol.columns)
variacion_sol = (uf_sol["valor"].iloc[-1] / uf_sol["valor"].iloc[0] - 1) * 100
print(f"Variación de la UF en la serie ({origen}): {variacion_sol:.2f}%")
uf_sol.tail()

---
## 4 · Ahora con SU dataset — plantilla de obtención reproducible (bloques 5–6)

Lo que sigue no es un ejercicio: es el **esqueleto real** del notebook `01-obtencion.ipynb` de su proyecto. Cópienlo a su repositorio y complétenlo hoy con su dataset aprobado. Esta estructura es la que se revisa en el Informe 1 (20-ago).

```
proyecto/
├── README.md                  ← metadata de la fuente (tabla de abajo)
├── data/raw/                  ← datos crudos: NO SE TOCAN
└── notebooks/01-obtencion.ipynb
```


In [ ]:
# ==========================================================
# PLANTILLA 01-obtencion — completar con su dataset aprobado
# ==========================================================

# --- 1. Metadata de la fuente (también va en el README) ---
METADATA = {
    "fuente": "",            # ej: "DEIS — Listas de espera GLOSA"
    "url": "",               # URL exacta de descarga o endpoint
    "fecha_descarga": "",    # ej: "2026-08-13"
    "licencia": "",          # ej: "Datos abiertos, uso libre con cita"
    "periodo_cubierto": "",  # ej: "2020–2026, corte junio 2026"
    "formato": "",           # ej: "CSV latin-1, sep=';'"
}

# --- 2. Descarga a data/raw/ (elegir la vía que corresponda) ---
# Vía archivo:  urllib.request.urlretrieve(METADATA["url"], "data/raw/archivo.csv")
# Vía API:      r = requests.get(...); guardar r.json() crudo a data/raw/
# Vía manual:   documentar aquí los pasos exactos (clics, filtros, URL, fecha)

# --- 3. Verificación básica (la "prueba de vida" del dataset) ---
# df = pd.read_csv("data/raw/archivo.csv", ...)   # con los parámetros correctos (Ej. 1)
# print(df.shape)          # ¿≥ ~1.000 filas y ≥ 8 columnas?
# df.info()                # ¿dtypes correctos?
# df.isna().sum()          # panorama de nulos (insumo del Informe 2)

print("Plantilla lista — completar con el dataset del equipo.")

### Checklist de salida de la Clase 2

- [ ] Propuesta **aprobada** (o ajustes aplicados / respaldo activado)
- [ ] Dataset principal en `data/raw/`, sin modificaciones
- [ ] Obtención automatizada o documentada paso a paso
- [ ] `METADATA` completa (y copiada al README)
- [ ] Verificación básica ejecutada: dimensiones, `info()`, nulos
- [ ] Si usaron API: respuesta cruda guardada a disco

**Encargo para la Clase 3 (20-ago):** obtención 100% reproducible terminada. El **Informe 1 se entrega ese mismo día a las 23:59** vía AULA.